<a href="https://colab.research.google.com/github/Akhilesh-2007/Traffic-demand-prediction/blob/main/Flipkart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Claude 90%

### Enhancing Model Performance: Weighted Ensembling and More Target Encoding

To try and achieve even higher predictive accuracy, we'll implement two main changes:

1.  **Weighted Averaging for Ensemble:** We'll introduce explicit weights for LightGBM and CatBoost predictions in the final ensemble. This allows for more flexible combination of the models' strengths. Initially, these weights will be equal, but you can adjust them.

2.  **Expanded Target Encoding:** We'll apply target encoding to additional features: `NumberofLanes` and `hour`. Target encoding can be powerful for categorical or pseudo-categorical features, as it replaces them with the mean of the target variable, effectively embedding target information directly into the feature space.

In [ ]:
# Define weights for the ensemble (can be tuned)
WEIGHT_LGB = 0.5
WEIGHT_CB = 0.5

# Ensure weights sum to 1 for proper averaging
TOTAL_WEIGHT = WEIGHT_LGB + WEIGHT_CB
WEIGHT_LGB /= TOTAL_WEIGHT
WEIGHT_CB /= TOTAL_WEIGHT

In [ ]:
# ==============================================================
# TRAFFIC DEMAND PREDICTION — SUBMISSION V2
# Expected Score: 94–97 on leaderboard
#
# KEY DESIGN:
#   - Trains on ALL rows (day 48 + day 49)
#   - Day 48 rows: f_ex = NaN  (no self-leakage)
#   - Day 49 rows: f_ex = actual d48 demand (strong lag feature)
#   - Test rows:   f_ex = actual d48 demand
#   - KFold target encoding + LightGBM 5-fold ensemble
# ==============================================================

# ── INSTALL (uncomment in Colab) ──────────────────────────────
# !pip install lightgbm -q
!pip install catboost -q

import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor # Added CatBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings("ignore")

# ==============================================================
# 1.  LOAD DATA
# ==============================================================
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")

for df in [train, test]:
    df["hour"]      = df["timestamp"].str.split(":").str[0].astype(int)
    df["minute"]    = df["timestamp"].str.split(":").str[1].astype(int)
    df["time_slot"] = df["hour"] * 4 + df["minute"] // 15

print(f"Train: {train.shape}  |  Test: {test.shape}")

# ==============================================================
# 2.  SPLIT INTO DAY 48 AND DAY 49
# ==============================================================
d48 = train[train["day"] == 48].copy().reset_index(drop=True)
d49 = train[train["day"] == 49].copy().reset_index(drop=True)

gm = train["demand"].mean()         # global mean (fallback)
mt = train["Temperature"].median()  # temperature fallback

print(f"Day 48 rows: {len(d48)}  |  Day 49 rows: {len(d49)}")

# ==============================================================
# 3.  BUILD DAY-48 LOOKUP DICTIONARIES
# ==============================================================
d48["geo_ts"] = d48["geohash"] + "_" + d48["timestamp"]
d48_exact     = d48.set_index("geo_ts")["demand"].to_dict()

d48_geo_hour_mean = d48.groupby(["geohash","hour"])["demand"].mean().to_dict()
d48_geo_hour_std  = d48.groupby(["geohash","hour"])["demand"].std().fillna(0).to_dict()
d48_geo_hour_min  = d48.groupby(["geohash","hour"])["demand"].min().to_dict()
d48_geo_hour_max  = d48.groupby(["geohash","hour"])["demand"].max().to_dict()

d48_geo_mean = d48.groupby("geohash")["demand"].mean().to_dict()
d48_geo_std  = d48.groupby("geohash")["demand"].std().fillna(0).to_dict()
d48_geo_min  = d48.groupby("geohash")["demand"].min().to_dict()
d48_geo_max  = d48.groupby("geohash")["demand"].max().to_dict()

d48_geo5_mean = d48.assign(g5=d48["geohash"].str[:5]).groupby("g5")["demand"].mean().to_dict()
d48_geo4_mean = d48.assign(g4=d48["geohash"].str[:4]).groupby("g4")["demand"].mean().to_dict()

d48_hour_mean = d48.groupby("hour")["demand"].mean().to_dict()
d48_slot_mean = d48.groupby("time_slot")["demand"].mean().to_dict()
d48_rt_hour   = d48.groupby(["RoadType","hour"])["demand"].mean().to_dict()
d48_ln_hour   = d48.groupby(["NumberofLanes","hour"])["demand"].mean().to_dict()

# ==============================================================
# 4.  FEATURE ENGINEERING
#     CRITICAL: use_exact=False for d48 rows  → f_ex = NaN
#               use_exact=True  for d49 + test → f_ex = d48 lag
#     Setting f_ex from the row's own demand would be 100% leakage.
#     The model learns to work both with and without the lag.
# ==============================================================
def build_features(df, use_exact: bool) -> pd.DataFrame:
    df  = df.copy()
    g5  = df["geohash"].str[:5]
    g4  = df["geohash"].str[:4]

    # ── d48 exact-match lag ──────────────────────────────────
    if use_exact:
        df["f_ex"] = (df["geohash"] + "_" + df["timestamp"]).map(d48_exact)
    else:
        df["f_ex"] = np.nan        # deliberately NaN for d48 rows
    df["f_has_ex"] = df["f_ex"].notna().astype(np.int8)

    # ── Geo × hour stats ─────────────────────────────────────
    gh = list(zip(df["geohash"], df["hour"]))
    df["f_gh_mean"] = [d48_geo_hour_mean.get(k, gm) for k in gh]
    df["f_gh_std"]  = [d48_geo_hour_std.get(k,   0) for k in gh]
    df["f_gh_min"]  = [d48_geo_hour_min.get(k,  0.) for k in gh]
    df["f_gh_max"]  = [d48_geo_hour_max.get(k,  1.) for k in gh]

    # ── Geo-level stats ──────────────────────────────────────
    df["f_g_mean"] = df["geohash"].map(d48_geo_mean).fillna(gm)
    df["f_g_std"]  = df["geohash"].map(d48_geo_std).fillna(0)
    df["f_g_min"]  = df["geohash"].map(d48_geo_min).fillna(0)
    df["f_g_max"]  = df["geohash"].map(d48_geo_max).fillna(1)

    # ── Coarser geo stats ────────────────────────────────────
    df["f_geo5"] = g5.map(d48_geo5_mean).fillna(gm)
    df["f_geo4"] = g4.map(d48_geo4_mean).fillna(gm)

    # ── Time-level stats ─────────────────────────────────────
    df["f_hour_mean"] = df["hour"].map(d48_hour_mean).fillna(gm)
    df["f_slot_mean"] = df["time_slot"].map(d48_slot_mean).fillna(gm)

    # ── Cross-feature stats ──────────────────────────────────
    df["f_rt_hour"] = [d48_rt_hour.get((r, h), gm)
                       for r, h in zip(df["RoadType"].fillna("Unknown"), df["hour"])]
    df["f_ln_hour"] = [d48_ln_hour.get((l, h), gm)
                       for l, h in zip(df["NumberofLanes"], df["hour"])]

    # ── Best available d48 estimate (cascade fallback) ───────
    df["f_best"] = (df["f_ex"]
                    .fillna(df["f_gh_mean"])
                    .fillna(df["f_g_mean"])
                    .fillna(gm))

    # ── Ratio features ───────────────────────────────────────
    df["f_ex_vs_hour"]  = df["f_ex"] / df["f_hour_mean"].clip(1e-9)
    df["f_ex_vs_gmean"] = df["f_ex"] / df["f_g_mean"].clip(1e-9)
    df["f_gh_vs_g"]     = df["f_gh_mean"] / df["f_g_mean"].clip(1e-9)

    # ── Range / spread ───────────────────────────────────────
    df["f_g_range"]  = df["f_g_max"]  - df["f_g_min"]
    df["f_gh_range"] = df["f_gh_max"] - df["f_gh_min"]

    # ── Interaction ──────────────────────────────────────────
    df["f_best_x_lanes"] = df["f_best"] * df["NumberofLanes"]
    df["f_best_x_temperature"] = df["f_best"] * df["Temperature"] # NEW interaction feature

    # ── Cyclical time encodings ──────────────────────────────
    df["hr_sin"] = np.sin(2 * np.pi * df["hour"]      / 24)
    df["hr_cos"] = np.cos(2 * np.pi * df["hour"]       / 24)
    df["mn_sin"] = np.sin(2 * np.pi * df["minute"]    / 60)
    df["mn_cos"] = np.cos(2 * np.pi * df["minute"]    / 60)
    df["sl_sin"] = np.sin(2 * np.pi * df["time_slot"] / 96)
    df["sl_cos"] = np.cos(2 * np.pi * df["time_slot"] / 96)
    df["is_peak"]= df["hour"].isin([7,8,9,10,11,12,13]).astype(np.int8)

    # ── Geohash prefix ───────────────────────────────────────
    df["geo5"] = g5
    df["geo4"] = g4

    # ── Fill categoricals ────────────────────────────────────
    for col in ["RoadType","LargeVehicles","Landmarks","Weather"]:
        df[col] = df[col].fillna("Unknown")
    # Corrected line to fill 'Temperature' NaNs with its own median, not from another column
    df["Temperature"] = df["Temperature"].fillna(mt)

    return df

# d48 → no exact lag (would be self-leakage)
d48_f  = build_features(d48,  use_exact=False)
# d49 → use d48 as lag (clean: different days)
d49_f  = build_features(d49,  use_exact=True)
# test → use d48 as lag
test_f = build_features(test, use_exact=True)

FEAT_COLS = [
    "hour","minute","time_slot","day",
    "f_ex","f_has_ex","f_best",
    "f_gh_mean","f_gh_std","f_gh_min","f_gh_max",
    "f_g_mean","f_g_std","f_g_min","f_g_max",
    "f_geo5","f_geo4","f_hour_mean","f_slot_mean",
    "f_rt_hour","f_ln_hour",
    "f_ex_vs_hour","f_ex_vs_gmean","f_gh_vs_g",
    "f_g_range","f_gh_range","f_best_x_lanes","f_best_x_temperature", # Added new feature
    "hr_sin","hr_cos","mn_sin","mn_cos","sl_sin","sl_cos","is_peak",
    "NumberofLanes","Temperature",
    "RoadType","LargeVehicles","Landmarks","Weather",
    "geohash","geo5","geo4",
]
CAT_COLS = ["RoadType","LargeVehicles","Landmarks","Weather","geohash","geo5","geo4"]

# ==============================================================
# 5.  COMBINE TRAIN  (d48 + d49)
# ==============================================================
tr_all = pd.concat([d48_f, d49_f], ignore_index=True)
X      = tr_all[FEAT_COLS]
y      = tr_all["demand"]
X_test = test_f[FEAT_COLS]

print(f"Combined train: {X.shape}  |  Test: {X_test.shape}")

# ==============================================================
# 6.  K-FOLD TARGET ENCODING
# ==============================================================
SM = 20
GM = y.mean()
KF = KFold(n_splits=5, shuffle=True, random_state=42)

# Expanded target encoding to include 'NumberofLanes' and 'hour'
for ec in ["geohash","geo5","geo4","RoadType","NumberofLanes","hour"]:
    col_name = f"te_{ec}"
    te_train = np.zeros(len(X))

    for _, (tr_i, vl_i) in enumerate(KF.split(X)):
        stats  = y.iloc[tr_i].groupby(X[ec].iloc[tr_i]).agg(["mean","count"])
        smooth = (stats["mean"]*stats["count"] + GM*SM) / (stats["count"] + SM)
        te_train[vl_i] = X[ec].iloc[vl_i].map(smooth).fillna(GM).values

    full_stats   = y.groupby(X[ec]).agg(["mean","count"])
    smooth_full  = (full_stats["mean"]*full_stats["count"] + GM*SM) / (full_stats["count"] + SM)

    tr_all[col_name]     = te_train
    test_f[col_name]     = X_test[ec].map(smooth_full).fillna(GM).values
    FEAT_COLS.append(col_name)
    print(f"  Target-encoded: {col_name}")

X      = tr_all[FEAT_COLS]
X_test = test_f[FEAT_COLS]

# Integer-encode categoricals for LightGBM and CatBoost
X_lgb  = X.copy()
Xt_lgb = X_test.copy()
X_cb = X.copy()
Xt_cb = X_test.copy()

for c in CAT_COLS:
    cats       = pd.Categorical(pd.concat([X[c], X_test[c]]))
    X_lgb[c]   = pd.Categorical(X[c],      categories=cats.categories).codes
    Xt_lgb[c]  = pd.Categorical(X_test[c], categories=cats.categories).codes.clip(0)

    # CatBoost handles categorical features directly, but needs them specified as int/str
    # For consistency, we will keep the original categorical values for CatBoost

print(f"\nFinal feature count: {X.shape[1]}")

# ==============================================================
# 7.  LIGHTGBM  5-FOLD TRAINING
# ==============================================================
lgb_params = dict(
    objective         = "regression_l1",
    metric            = "rmse",
    learning_rate     = 0.025, # Slightly reduced for more stable learning with more rounds
    num_leaves        = 1023,  # Increased for higher model complexity
    max_depth         = -1,
    min_child_samples = 10,
    feature_fraction  = 0.75,
    bagging_fraction  = 0.75,
    bagging_freq      = 5,
    lambda_l1         = 0.05,
    lambda_l2         = 0.05,
    seed              = 42,
    verbose           = 100,
    device            = "gpu",
)

KFT     = KFold(n_splits=5, shuffle=True, random_state=42)
oof_lgb     = np.zeros(len(X_lgb))
te_pred_lgb = np.zeros(len(Xt_lgb))

# Create a LightGBM specific categorical features list, excluding 'geohash'
lgb_cat_cols = [col for col in CAT_COLS if col != 'geohash']

print("\n🚀 Starting LightGBM Training...")
for fold, (tr_i, vl_i) in enumerate(KFT.split(X_lgb)):
    print(f"  LightGBM Fold {fold+1}/5 ...")
    model_lgb = lgb.train(
        lgb_params,
        lgb.Dataset(X_lgb.iloc[tr_i], label=y.iloc[tr_i], categorical_feature=lgb_cat_cols),
        num_boost_round = 8000, # Increased max boosting rounds
        valid_sets      = [lgb.Dataset(X_lgb.iloc[vl_i], label=y.iloc[vl_i], categorical_feature=lgb_cat_cols)],
        callbacks       = [
            lgb.early_stopping(stopping_rounds=200, verbose=False), # Increased early stopping rounds
            lgb.log_evaluation(period=100),
        ],
    )
    oof_lgb[vl_i]  = model_lgb.predict(X_lgb.iloc[vl_i])
    te_pred_lgb   += model_lgb.predict(Xt_lgb) / 5

# ==============================================================
# 8.  CATBOOST  5-FOLD TRAINING (Added)
# ==============================================================
cb_params = dict(
    iterations        = 5000, # Increased Number of boosting rounds
    learning_rate     = 0.03, # Slightly reduced for more stable learning with more iterations
    depth             = 10, # Increased for deeper trees
    l2_leaf_reg       = 5,
    loss_function     = "RMSE", # Using RMSE for CatBoost too
    eval_metric       = "R2",   # Evaluate using R2
    random_seed       = 42,
    verbose           = 100,
    early_stopping_rounds = 150, # Increased early stopping rounds
    task_type         = "GPU", # Enable GPU for CatBoost
)

oof_cb     = np.zeros(len(X_cb))
te_pred_cb = np.zeros(len(Xt_cb))

print("\n🚀 Starting CatBoost Training...")
for fold, (tr_i, vl_i) in enumerate(KFT.split(X_cb)):
    print(f"  CatBoost Fold {fold+1}/5 ...")
    model_cb = CatBoostRegressor(
        **cb_params,
        cat_features=[X_cb.columns.get_loc(col) for col in CAT_COLS if col in X_cb.columns]
    )
    model_cb.fit(
        X_cb.iloc[tr_i], y.iloc[tr_i],
        eval_set=(X_cb.iloc[vl_i], y.iloc[vl_i]),
        early_stopping_rounds=cb_params['early_stopping_rounds'],
        verbose=False # Set to False to avoid verbose output during fit in each fold, will use verbose from params
    )
    oof_cb[vl_i]  = model_cb.predict(X_cb.iloc[vl_i])
    te_pred_cb   += model_cb.predict(Xt_cb) / 5

# ==============================================================
# 9.  ENSEMBLING AND FINAL EVALUATION (Modified)
# ==============================================================
# Weighted average ensemble
oof_ensemble = (oof_lgb * WEIGHT_LGB + oof_cb * WEIGHT_CB)
te_pred_ensemble = (te_pred_lgb * WEIGHT_LGB + te_pred_cb * WEIGHT_CB)

print("\n--- LightGBM Model Metrics ---")
oof_r2_lgb = r2_score(y, oof_lgb)
print(f"✅ Overall OOF R2 (LGB): {oof_r2_lgb:.5f}  ({oof_r2_lgb*100:.2f})")
oof_mae_lgb = mean_absolute_error(y, oof_lgb)
oof_mse_lgb = mean_squared_error(y, oof_lgb)
oof_rmse_lgb = np.sqrt(oof_mse_lgb)
print(f"✅ Overall OOF MAE (LGB): {oof_mae_lgb:.5f}")
print(f"✅ Overall OOF MSE (LGB): {oof_mse_lgb:.5f}")
print(f"✅ Overall OOF RMSE (LGB): {oof_rmse_lgb:.5f}")

idx49  = tr_all.index[tr_all["day"] == 49].tolist()
d49_r2_lgb = r2_score(y.iloc[idx49], oof_lgb[idx49])
print(f"✅ Day-49 OOF R2 (LGB) : {d49_r2_lgb:.5f}  ({d49_r2_lgb*100:.2f})")
d49_mae_lgb = mean_absolute_error(y.iloc[idx49], oof_lgb[idx49])
d49_mse_lgb = mean_squared_error(y.iloc[idx49], oof_lgb[idx49])
d49_rmse_lgb = np.sqrt(d49_mse_lgb)
print(f"✅ Day-49 OOF MAE (LGB): {d49_mae_lgb:.5f}")
print(f"✅ Day-49 OOF MSE (LGB): {d49_mse_lgb:.5f}")
print(f"✅ Day-49 OOF RMSE (LGB): {d49_rmse_lgb:.5f}")

print("\n--- CatBoost Model Metrics ---")
oof_r2_cb = r2_score(y, oof_cb)
print(f"✅ Overall OOF R2 (CB): {oof_r2_cb:.5f}  ({oof_r2_cb*100:.2f})")
oof_mae_cb = mean_absolute_error(y, oof_cb)
oof_mse_cb = mean_squared_error(y, oof_cb)
oof_rmse_cb = np.sqrt(oof_mse_cb)
print(f"✅ Overall OOF MAE (CB): {oof_mae_cb:.5f}")
print(f"✅ Overall OOF MSE (CB): {oof_mse_cb:.5f}")
print(f"✅ Overall OOF RMSE (CB): {oof_rmse_cb:.5f}")

d49_r2_cb = r2_score(y.iloc[idx49], oof_cb[idx49])
print(f"✅ Day-49 OOF R2 (CB) : {d49_r2_cb:.5f}  ({d49_r2_cb*100:.2f})")
d49_mae_cb = mean_absolute_error(y.iloc[idx49], oof_cb[idx49])
d49_mse_cb = mean_squared_error(y.iloc[idx49], oof_cb[idx49])
d49_rmse_cb = np.sqrt(d49_mse_cb)
print(f"✅ Day-49 OOF MAE (CB): {d49_mae_cb:.5f}")
print(f"✅ Day-49 OOF MSE (CB): {d49_mse_cb:.5f}")
print(f"✅ Day-49 OOF RMSE (CB): {d49_rmse_cb:.5f}")

print("\n--- Ensemble Model Metrics ---")
oof_r2_ensemble = r2_score(y, oof_ensemble)
print(f"✅ Overall OOF R2 (Ensemble): {oof_r2_ensemble:.5f}  ({oof_r2_ensemble*100:.2f})")
oof_mae_ensemble = mean_absolute_error(y, oof_ensemble)
oof_mse_ensemble = mean_squared_error(y, oof_ensemble)
oof_rmse_ensemble = np.sqrt(oof_mse_ensemble)
print(f"✅ Overall OOF MAE (Ensemble): {oof_mae_ensemble:.5f}")
print(f"✅ Overall OOF MSE (Ensemble): {oof_mse_ensemble:.5f}")
print(f"✅ Overall OOF RMSE (Ensemble): {oof_rmse_ensemble:.5f}")

d49_r2_ensemble = r2_score(y.iloc[idx49], oof_ensemble[idx49])
print(f"✅ Day-49 OOF R2 (Ensemble) : {d49_r2_ensemble:.5f}  ({d49_r2_ensemble*100:.2f})")
d49_mae_ensemble = mean_absolute_error(y.iloc[idx49], oof_ensemble[idx49])
d49_mse_ensemble = mean_squared_error(y.iloc[idx49], oof_ensemble[idx49])
d49_rmse_ensemble = np.sqrt(d49_mse_ensemble)
print(f"✅ Day-49 OOF MAE (Ensemble): {d49_mae_ensemble:.5f}")
print(f"✅ Day-49 OOF MSE (Ensemble): {d49_mse_ensemble:.5f}")
print(f"✅ Day-49 OOF RMSE (Ensemble): {d49_rmse_ensemble:.5f}")


# ==============================================================
# 10. SUBMISSION (Modified to use ensemble predictions)
# ==============================================================
final_preds = np.clip(te_pred_ensemble, 0.0, 1.0)

submission = pd.DataFrame({
    "Index" : test["Index"],
    "demand": final_preds,
})
submission.to_csv("submission.csv", index=False)

print(f"\n✅ submission.csv saved!")
print(f"   Rows  : {len(submission)}")
print(f"   Range : [{final_preds.min():.5f}, {final_preds.max():.5f}]")

# ==============================================================
# 11. FEATURE IMPORTANCE (Showing LightGBM's importance for now)
# ==============================================================
fi = pd.DataFrame({
    "Feature"   : X_lgb.columns,
    "Importance": model_lgb.feature_importance(),
}).sort_values("Importance", ascending=False)

print("\n📊 Top 20 LightGBM Features:")
print(fi.head(20).to_string(index=False))